# Run pipeline
Run the Seq2Pocket pipeline for PDB ID: 4GQQ, chain ID: A. You have to download the models first and install packages from `requirements.txt`.

In [ ]:
from transformers import AutoTokenizer
import torch
import sys

# ACTION NEEDED: Update PATH, MODEL_PATH, and SMOOTHING_MODEL_PATH!
PATH = '/path/to/seq2pocket'  
MODEL_PATH = f"{PATH}/path/to/gbs-model-enhanced-scPDB-filtered.pt"
SMOOTHING_MODEL_PATH = f'{PATH}/path/to/smoother.pt'

sys.path.append(f'{PATH}/tutorial')
import finetuning_utils
from finetuning_utils import FinetunedEsmModel

DROPOUT = 0.3
OUTPUT_SIZE = 1

MODEL_NAME = 'facebook/esm2_t36_3B_UR50D'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_LENGTH = 1024
DECISION_THRESHOLD = 0.7

### Run finetuned model
Get probabilities from the finetuned model.

In [2]:
# load the model
model = torch.load(MODEL_PATH, weights_only=False)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model.eval()

sequence = 'EYSPNTQQGRTSIVHLFEWRWVDIALECERYLAPKGFGGVQVSPPNENVAIYNPFRPWWERYQPVSYKLCTRSGNEDEFRNMVTRCNNVGVRIYVDAVINHMCGNAVSAGTSSTCGSYFNPGSRDFPAVPYSGWDFNDGKCKTGSGDIENYNDATQVRDCRLTGLLDLALEKDYVRSKIAEYMNHLIDIGVAGFRLDASKHMWPGDIKAILDKLHNLNSNWFPAGSKPFIYQEVIDLGGEPIKSSDYFGNGRVTEFKYGAKLGTVIRKWNGEKMSYLKNWGEGWGFVPSDRALVFVDNHDNQRGHGAGGASILTFWDARLYKMAVGFMLAHPYGFTRVMSSYRWPRQFQNGNDVNDWVGPPNNNGVIKEVTINPDTTCGNDWVCEHRWRQIRNMVIFRNVVDGQPFTNWYDNGSNQVAFGRGNRGFIVFNNDDWSFSLTLQTGLPAGTYCDVISGDKINGNCTGIKIYVSDDGKAHFSISNSAEDPFIAIHAESKL'

# tokenize the sequence
tokenized_sequences = tokenizer(sequence, max_length=MAX_LENGTH, padding='max_length', truncation=True)
tokenized_sequences = {k: torch.tensor([v]).to(DEVICE) for k,v in tokenized_sequences.items()}

# predict
output = model(tokenized_sequences)
# for CBS model which returns tuple
if isinstance(output, tuple):
    output = output[0]

output = output.flatten()

mask = (tokenized_sequences['attention_mask'] == 1).flatten()

probabilities = torch.sigmoid(output[mask][1:-1]).detach().cpu().numpy()
predictions = (probabilities > DECISION_THRESHOLD).astype(float)
print(predictions)


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.
 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 1. 0. 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 1. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

## Generate embeddings
Create embeddings for the Smoothing classifier.

In [3]:
import esm
import numpy as np

# load model & tokenizer
LAST_LAYER_INDEX = 36
model, alphabet = esm.pretrained.esm2_t36_3B_UR50D()
batch_converter = alphabet.get_batch_converter()

# tokenize
data = [("seq", sequence)]
_, _, batch_tokens = batch_converter(data)

# to GPU if available
batch_tokens = batch_tokens.to(DEVICE)
model.to(DEVICE)

# generate & save embedding
with torch.no_grad():
    results = model(batch_tokens, repr_layers=[LAST_LAYER_INDEX], return_contacts=True)
    token_representations = results["representations"][LAST_LAYER_INDEX]
    vector = token_representations.detach().cpu().numpy()[0][1:-1]

embedding_path = f'{PATH}/tutorial/esm_vector.npy'
np.save(embedding_path, vector)

## Run Smoothing
Run Smoothing classifier.

In [4]:
sys.path.append(f'{PATH}/src/utils')
import eval_utils
from eval_utils import CryptoBenchClassifier

POSITIVE_DISTANCE_THRESHOLD = 15

# load the smoothing model
smoothing_model = torch.load(SMOOTHING_MODEL_PATH, weights_only=False) #, map_location=torch.device('cpu'))

# load the embedding and coordinates
embedding = np.load(embedding_path)
coordinates_path = f'{PATH}/tutorial/4gqqA.npy'
coordinates = np.load(coordinates_path)
distance_matrix = eval_utils.compute_distance_matrix(coordinates)

assert embedding.shape[0] == distance_matrix.shape[0], "The number of residues in the embedding and distance matrix must be the same."
assert len(sequence) == embedding.shape[0], "The number of residues in the sequence and embedding must be the same."

# consider each negatively positive residue for smoothing
for residue_idx in np.where(predictions == 0.0)[0]:
    current_residue_embedding = embedding[residue_idx]
    close_residues_indices = np.where(distance_matrix[residue_idx] < POSITIVE_DISTANCE_THRESHOLD)[0]
    # get the close binding residues
    close_residues_indices = np.where(distance_matrix[residue_idx] < POSITIVE_DISTANCE_THRESHOLD)[0]
    close_binding_residues_indices = np.intersect1d(close_residues_indices, np.where(predictions == 1.0)[0])
    # create embedding 
    if len(close_binding_residues_indices) == 0:
        # no close binding residues - skip this residue
        continue
    elif len(close_binding_residues_indices) == 1:
        surrounding_embedding = embedding[close_binding_residues_indices].reshape(-1)
    else:
        # get the mean of the close binding residues
        surrounding_embedding = np.mean(embedding[close_binding_residues_indices], axis=0).reshape(-1)

    concatenated_embedding = torch.tensor(np.concatenate((current_residue_embedding, surrounding_embedding), axis=0), dtype=torch.float32).to(DEVICE)
        
        # get the prediction
    test_logits = smoothing_model(concatenated_embedding).squeeze()
    result = (torch.sigmoid(test_logits)>eval_utils.SMOOTHING_DECISION_THRESHOLD).float()
    if result == 1:
        # set the residue as binding
        predictions[residue_idx] = 1
        print(f"Residue {residue_idx} was changed to binding by the smoothing model.")

print()
print("Predictions after smoothing:")
print(predictions)

Residue 18 was changed to binding by the smoothing model.
Residue 140 was changed to binding by the smoothing model.
Residue 147 was changed to binding by the smoothing model.
Residue 156 was changed to binding by the smoothing model.
Residue 159 was changed to binding by the smoothing model.
Residue 160 was changed to binding by the smoothing model.
Residue 162 was changed to binding by the smoothing model.
Residue 165 was changed to binding by the smoothing model.
Residue 166 was changed to binding by the smoothing model.
Residue 167 was changed to binding by the smoothing model.
Residue 169 was changed to binding by the smoothing model.
Residue 233 was changed to binding by the smoothing model.
Residue 235 was changed to binding by the smoothing model.
Residue 239 was changed to binding by the smoothing model.
Residue 256 was changed to binding by the smoothing model.
Residue 298 was changed to binding by the smoothing model.
Residue 301 was changed to binding by the smoothing model